In [1]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

/tmp/ipykernel_61829/3777615979.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


# 📘 CityAI – Multiplicació de matrius i Algoritme d'Strassen

La ciutat de Barcelona vol publicar en temps real les imatges de les càmeres de trànsit que hi ha per la ciutat, perquè els ciutadans sàpiguen l'estat de les zones de la ciutat que solen estar més congestionades. A la vegada vol assegurar-se que respecta la privacitat tant dels vianants com dels conductors de vehicles que apareixen en aquestes imatges, així que ha decidit que difuminarà totes les cares i matrícules que apareguin a les imatges.

Per a fer-ho, li cal desenvolupar un programari que permeti aplicar filtres a les imatges de les càmeres i, per tant, que pugui multiplicar matrius grans de manera eficient.

## 🧠 Objectius

Aprofundir en els diferents algoritmes de multiplicació de matrius: força bruta, dividir i vèncer, i l'algoritme de Strassen.

## Multiplicació de matrius amb força bruta

L'algorisme de **força bruta** per a la multiplicació de matrius consisteix a calcular cada element de la matriu resultant aplicant la fórmula de multiplicació de matrius.

Donades dues matrius:

- $(A)$, de dimensions $(m * n)$,
- $(B)$, de dimensions $(n * p)$,

la matriu resultant $(C)$, de dimensions $(m * p)$, es calcula com:

$ C[i][j] = \sum_{k=0}^{n-1} A[i][k] * B[k][j]$

![Multiplicació de matrius tradicional](img/matmul.png "Multiplicació de matrius tradicional")


Per exemple:  
  
$${
\left[
  \begin{array}{cccc}
    3 & 5 & 1 & 3 \\
    1 & 2 & 3 & 4 \\
    4 & 5 & 6 & 8 \\
    7 & 8 & 3 & 3 \\  
  \end{array}
\right]
\cdot
\left[
  \begin{array}{cccc}
    4 & 1 & 2 & 3 \\
    1 & 2 & 1 & 6 \\
    2 & 4 & 6 & 2 \\
    6 & 2 & 5 & 4 \\  
  \end{array}
\right]
= 
\left[
  \begin{array}{cccc}
    37 & 23 & 32 & 53 \\
    36 & 25 & 42 & 37 \\
    81 & 54 & 89 & 86 \\
    72 & 65 & 91 & 99 \\  
  \end{array}
\right]
}
$$
  
  

In [2]:
# Utilitzarem com a suport la llibreria numpy, que es pot instal·lar com a paquet de Python, si no està ja instal·lat
# !pip3 install numpy

In [19]:
import numpy as np

def forca_bruta(A, B):
    """Multiplicació de dues matrius utilitzant el mètode de força bruta.

    Parameters
    ----------
    A: Matriu de mida n x m
    B: Matriu de mida m x p

    Returns
    -------
    C: Matriu de mida n x p
    """
    # Agafem les mides de les matrius
    n, m, p = A.shape[0], A.shape[1], B.shape[1]
    
    # Creem la matriu de sortida C
    C = np.array([[0] * p for i in range(n)])
    
    # Apliquem la fòrmula per calcular cada posició de C
    for i in range(n):
        for j in range(p):
            for k in range(m):
                C[i][j] += A[i][k] * B[k][j]
    
    return C

Comprovo que la multiplicació és correcta usant la funció **np.array_equal(matriu_A, matriu_B)**, que permet comparar matrius. Aquesta funció retorna **True** si les matrius són iguals i **False** si són diferents.

In [16]:
# Utilitzo d'entrada les matrius de l'exemple
X  = np.array([[3, 5, 1, 3],
              [1, 2, 3, 4],
              [4, 5, 6, 8],
              [7, 8, 9, 3]])

Y  = np.array([[4, 1, 2, 3],
              [1, 2, 1, 6],
              [2, 4, 6, 2],
              [6, 2, 5, 4]])

In [20]:
assert np.array_equal(forca_bruta(X, Y), np.array([[37, 23, 32, 53],
                                                   [36, 25, 42, 37],
                                                   [81, 54, 89, 86],
                                                   [72, 65, 91, 99]]))

### Complexitat del mètode de força bruta

Si ens fixem en el codi de força bruta, veurem que tenim 3 estructures iteratives aniuades. Suposant el producte de matrius **quadrades** de mida n (n == m == p), podem veure que el mètode de força bruta té complexitat $\mathcal{O}(n^3)$.

## Multiplicació de matrius amb dividir i vèncer?

Les propietats del producte de matrius ens permeten utilizar l'estratègia de dividir i vèncer per a calcular el producte de dues matrius **quadrades** grans de mida ($n \times n$).

Aquesta és l'operació que faríem per multiplicar dues matrius si els seus elements (A, B, C, D, E, F, G, H) fossin números:

$$
X = 
\left[
  \begin{array}{cc}
    A & B  \\
    C & D  \\
  \end{array}
\right] \, , \;
Y = \left[
  \begin{array}{cc}
    E & F  \\
    G & H  \\  
  \end{array}
\right]
$$


$$ XY = 
\left[
  \begin{array}{cc}
    A & B  \\
    C & D  \\
  \end{array}
\right] \, \left[
  \begin{array}{cc}
    E & F  \\
    G & H  \\  
  \end{array}
\right] =
\left[
  \begin{array}{cc}
    AE+BG & AF+BH  \\
    CE+DG & CF+DH  \\  
  \end{array}
\right]
$$

Però resulta que aquesta operació també és vàlida si (A, B, C, D, E, F, G, H) són també matrius.

Això ens permet usar un algoritme de dividir i vèncer si anem separant de manera recursiva cada matriu en 4 matrius més petites ($\frac{n}{2}$, $\frac{n}{2}$) corresponent als 4 quadrants.

![Exemple de descomposició matrius en quadrants](img/StrassenBlocsExample1.png "Descomposició del producte de dues matrius")

Fixeu-vos que descomposem el producte de dues matrius ($n \times n$) en 8 productes de matrius $\frac{n}{2}$ x $\frac{n}{2}$) i 4 sumes.

Per calcular el primer terme $a \times e$, el podem tornar a dividir de forma recursiva:

![Exemple de descomposició matrius en quadrants](img/StrassenBlocsExample3.png "Descomposició del producte de dues matrius")


## Exercici 1: Completa el codi del producte de matrius recursiu separant per blocs

In [21]:
def divisio_blocs(matrix):
    """Dividir una matriu n x n en 4 submatrius n/2 x n/2

    Parameters
    ----------
    matrix: Matriu de mida n x n
        
    Returns
    -------
    n1, n2, n3, n4: Matrius de mida n/2 x n/2
    """
    # Agafem la mida de la matriu quadrada
    n = len(matrix)
    
    # Generem els quatre blocs
    return matrix[:n//2, :n//2], matrix[:n//2, n//2:], matrix[n//2:, :n//2], matrix[n//2:, n//2:]

In [22]:
def multiplicacio_blocs(A, B):
    """Multiplicació de dues matrius n x n a partir de la divisió en quadrants

    Parameters
    ----------
    A: Matriu de mida n x n
    B: Matriu de mida n x n
       
    Returns
    -------
    C: Matriu de mida n x n
    """
    # Apliquem força bruta al cas trivial
    if len(A) <= 2:
        return forca_bruta(A, B)
    
    # Dividim les matrius en els 4 blocs
    a, b, c, d = divisio_blocs(A)
    e, f, g, h = divisio_blocs(B)

    # Calculem els diferents components
    ae = multiplicacio_blocs(a, e)
    bg = multiplicacio_blocs(b, g)
    af = multiplicacio_blocs(a, f)
    bh = multiplicacio_blocs(b, h)
    ce = multiplicacio_blocs(c, e)
    dg = multiplicacio_blocs(d, g)
    cf = multiplicacio_blocs(c, f)
    dh = multiplicacio_blocs(d, h)

    # Calculem el resultat per a cada quadrant
    C11 = ae + bg
    C12 = af + bh
    C21 = ce + dg
    C22 = cf + dh

    # Muntem la matriu final
    # (vstack apila les matrius verticalment en una nova matriu, i hstack horitzontalment)
    C = np.vstack(
        (
            np.hstack((C11, C12)), 
            np.hstack((C21, C22))
        ))
    
    return C

In [23]:
# Comprovo que la multiplicació amb aquest mètode és correcta
assert np.array_equal(multiplicacio_blocs(X, Y), np.array([[37, 23, 32, 53],
                                                           [36, 25, 42, 37],
                                                           [81, 54, 89, 86],
                                                           [72, 65, 91, 99]]))

### Complexitat del mètode de multiplicació dividint per blocs

Si ens fixem en el codi recursiu de multiplicació per blocs, veiem que hem descomposat el producte de dues matrius $n \times n$ en 8 productes de matrius ($\frac{n}{2} \times \frac{n}{2}$) i 4 sumes. A més, tenim la part de dividir la matriu i tornar-la a muntar. 

Aplicant el Teorema Mestre amb els següents paràmetres:
- $a = 8  \rightarrow$ Nombre de subproblemes o crides recursives.
- $b = 2 \rightarrow$ Factor de divisió de les files/columnes (factor n/2).
- $d = 2 \rightarrow$ Complexitat del treball fora de la recursió. La reconstrucció de la solució són $n^2$ sumes $O(n^2)$.

Veiem que la complexitat de la funció de recurrència d'aquest algorisme és:
$$
T(n)=a\,T\!\left(\frac{n}{b}\right)+\mathcal{O}(n^d)
\quad \Longrightarrow \quad
T(n)=8\,T\!\left(\frac{n}{2}\right)+\mathcal{O}(n^2).
$$

On el primer terme, $8T(n/2)$, representa les 8 multiplicacions recursives de submatrius, i el segon terme, $O(n^2)$, representa les operacions de suma/resta per combinar els resultats.

Com que $a > b^d$ (concretament, $8 > 2^2$), domina el pes dels subproblemes i la complexitat és $T(n) = \mathcal{O} \left( n^{\log_b a} \right)$. Com a conseqüència obtenim una complexitat de $\mathcal{O}(n^3)$, **la mateixa** que la obtinguda amb la multiplicació convencional de matrius per **força bruta**.

## Multiplicació de matrius amb l'Algoritme d'Strassen (dividir i vèncer)

Strassen es va adonar que reformulant el càlcul dels quadrants,  el producte de dues matrius es podia escriure d'una manera més simplificada:

$\begin{align}
        ae+bg &= (ae+bg) + (ah-ah) +(de-de) +(dh -dh) +(dg-dg) + (bh -bh)  \\ 
              &= \left( a+d \right) \left(e+h \right)+d \left( g-e \right)- \left(a+b\right)h+\left(b-d \right)\left( g+h \right) \\
    \end{align}$

$\begin{align}
        af+bh &= (af+bh) + (ah-ah) \\
              &= a \left( f-h \right) + \left( a+b \right) h \\
    \end{align}$

$\begin{align}
        ce+dg &= (ce+dg) + (de-de) \\
              &= \left( c+d \right) e + d \left( g-e \right) \\
    \end{align}$

$\begin{align}
        cf+dh &= (cf+dh) + (ae-ae) + (af-af) + (ah-ah) + (ce-ce) +(de-de) \\
              &= a \left( f-h \right) + \left(a+d \right) \left( e+h \right)- \left(c+d\right) e -\left(a-c \right)\left( e+f \right) \\
    \end{align}$
    
Si definim els diferents productes que es repeteixen en el càlcul dels diferents quadrants, podem expressar el càlcul en funció d'aquests productes:

${\left\{ \; \begin{align}
p_1 &= \left( a+d \right) \left(e+h \right) \\
p_2 &= d \left( g-e \right) \\
p_3 &= \left(a+b\right) h \\
p_4 &= \left(b-d \right)\left( g+h \right) \\
p_5 &= a \left( f-h \right) \\
p_6 &= \left( c+d \right) e \\
p_7 &= \left(a-c \right)\left( e+f \right) \\
\end{align} \right.}$


${\left\{ \; \begin{align}
ae+bg &= p1 + p2 - p3 + p4 \\
af+bh &= p5+p3 \\
ce+dg &= p6+p2 \\
cf+dh &= p5+p1-p6-p7 \\
\end{align} \right.}$

Així que l'Algoritme d'Strassen utilitza la mateixa estratègia de dividir i vèncer que hem vist a l'apartat anterior per fer la multiplicació de dues matrius **quadrades**, però en comptes de realitzar **vuit multiplicacions** en cada pas en realitza **només set**.

## Exercici 2: Completa el codi del producte de matrius recursiu separant per blocs

In [24]:
def strassen(A, B):
    """Multiplicació de dues matrius n x n a partir de la divisió en quadrants.

    Parameters
    ----------
    A: Matriu de mida n x n
    B: Matriu de mida n x n
        
    Returns
    -------
    C: Matriu de mida n x n
    """
    # Apliquem força bruta al cas trivial
    if len(A) <= 2:
        return forca_bruta(A, B)
    
    # Dividim les matrius en els 4 blocs
    a, b, c, d = divisio_blocs(A)
    e, f, g, h = divisio_blocs(B)

    # Calculem els diferents productes
    p1 = strassen(a+d, e+h)
    p2 = strassen(d, g-e)
    p3 = strassen(a+b, h)
    p4 = strassen(b-d, g+h)
    p5 = strassen(a, f-h)
    p6 = strassen(c+d, e)
    p7 = strassen(a-c, e+f)

    # Calculem el resultat per a cada quadrant
    C11 = p1 + p2 - p3 + p4
    C12 = p5 + p3
    C21 = p6 + p2
    C22 = p5 + p1 - p6 - p7

    # Muntem la matriu final
    # (vstack apila les matrius verticalment en una nova matriu, i hstack horitzontalment)
    C = np.vstack(
        (
            np.hstack((C11, C12)), 
            np.hstack((C21, C22))
        ))

    return C

In [25]:
# Comprovo que la multiplicació amb aquest mètode és correcta
assert np.array_equal(strassen(X, Y), np.array([[37, 23, 32, 53],
                                                [36, 25, 42, 37],
                                                [81, 54, 89, 86],
                                                [72, 65, 91, 99]]))

### Complexitat de la multiplicació usant l'algoritme d'Strassen

Si ens fixem en el codi recursiu de multiplicació usant l'algoritme d'Strassen, veiem que hem descomposat el producte de dues matrius $n \times n$ en 7 productes de matrius ($\frac{n}{2} \times \frac{n}{2}$) en comptes de 8. 

Aplicant el Teorema Mestre veiem que si repetim el càlcul de la complexitat per la funció de recurrència d'aquest algorisme ens dóna:  
    $\qquad T(n) = aT(n/b)+\mathcal{O}(n^d) \quad  \rightarrow \quad T(n) = 7T(n/2)+\mathcal{O}(n^2)$.

Aquí igual que abans $a > b^d$ (concretament, $7 > 2^2$), segueix dominant el pes dels subproblemes, i la complexitat és $T(n) = \mathcal{O} \left( n^{\log_b a} \right)$, però ara la complexitat resultant és de $\mathcal{O}(n^{2.81})$, lleugerament inferior a l'anterior. Tot i que pot semblar una diferència petita, en realitat **la millora és molt considerable**, especialment per a matrius grans.


## Exercici 3: Compara les complexitats pels dos algoritmes

Compara la diferència de complexitat entre els dos algoritmes quan la mida de les matrius és $n = 1000$ i quan és $n=1000000$. Quin algoritme és més ràpid? Amb quin factor?

In [32]:
print(1000 ** 3, 1000 ** 2.81)
print(1000000 ** 3, 1000000 ** 2.81)
print((1000000 ** 2.81) / (1000000 ** 3) * 100)

1000000000 269153480.3926917
1000000000000000000 7.244359600749906e+16
7.244359600749906


*L'algoritme d'Strassen és més ràpid, especifícament, 7.24% més ràpid*

## Limitacions de l'Algorisme de Strassen

Strassen té algunes limitacions
- **Sobrecàrrega de memòria**: Necessita emmagatzemar moltes submatrius durant els càlculs.
- **Eficàcia pràctica**: Per a dimensions petites de matrius, la sobrecàrrega de la divisió i combinació pot superar els beneficis del mètode.

## Crèdits

Aquesta llibreta s'ha el·laborat utilitzant les següents fonts d'informació:
- Materials de l'assignatura d'Algorismica (Jordi Vitrià i Mireia Ribera)
- Presentació i repositori GitHub Inside Code by Syphax Ait Outbelli:
    - Repositori (https://gist.github.com/syphhhttps://gist.github.com/syphh)
    - Video explicatiu (https://www.youtube.com/watch?v=OSelhO6Qnlc)